# GUI Component Prediction — Paper Table 2

Component-level counterpart to `_results_sequence_prediction.ipynb` (Table 3).
Run the cells top to bottom; step 2 needs a GPU, the rest run anywhere.

In [1]:
# 1. Extract components + text vocabulary from all RICO screens (~5 min)

In [2]:
%run 10_extract_components.py --out_dir component_data


[Step 1/3] Scanning '../rico-data/data/layouts'...
  66261 screens to extract

[Step 2/3] Extracting components (n_jobs=-1)...


Extracting screens: 100%|██████████| 259/259 [00:24<00:00, 10.55it/s]



[Step 3/3] Building vocabulary and writing dataset...

  Screens extracted        : 66092
  Components               : 1518766
  ...with non-empty text   : 613261
  Vocabulary size          : 198263
  Components per screen    : 22.98 (9.28 with text)

Wrote component_data\components.jsonl.gz
      component_data\vocab.json
      component_data\meta.json
Total extraction time: 39.0s

Next step — embed the vocabulary with:
  python 11_embed_component_vocab.py --data_dir component_data


In [3]:
# 2. Embed the vocabulary with Qwen3-VL-Embedding-8B (GPU; run on the cluster)
# CPU fallback:
# %run 11_embed_component_vocab.py --data_dir component_data --model sentence-transformers/all-MiniLM-L6-v2 --batch_size 512

In [4]:
# GPU
# %run 11_embed_component_vocab.py --data_dir component_data --batch_size 64

# CPU Fallback
%run 11_embed_component_vocab.py --data_dir component_data --model sentence-transformers/all-MiniLM-L6-v2 --batch_size 512

C:\Users\Jonathan\Dev\screen2vec-revisited\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[Step 1/3] Loading vocabulary from 'component_data\vocab.json'...
  198263 unique component texts

[Step 2/3] Loading encoder 'sentence-transformers/all-MiniLM-L6-v2' on cpu (pooling=mean)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2836.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Step 3/3] Encoding in batches of 512...


Embedding vocab: 100%|██████████| 388/388 [20:13<00:00,  3.13s/it]



  Embedding matrix: (198263, 384) (float16)
Wrote component_data\vocab_emb.npy
Total embedding time: 1216.8s

Next step — train the predictor with:
  python 12_train_component_predictor.py --data_dir component_data


In [5]:
# 3. EUCLIDEAN context — the first row of Table 2

In [ ]:
%run 12_train_component_predictor.py --data_dir component_data --distance euclidean --plot

Device: cpu

[Step 1/5] Loading component data from 'component_data'...
  Screens: 66092  |  Vocabulary: 198263
  Vocabulary embeddings: (198263, 384) from 'component_data\vocab_emb.npy'

[Step 2/5] Preparing the prediction space (pred_dim=512)...
  Prediction space: 384d -> 384d (unchanged)

[Step 3/5] Building euclidean contexts (n=16)...


Building euclidean contexts: 100%|██████████| 130/130 [00:11<00:00, 11.63it/s]


  Usable screens: 59751  |  Predictable components: 613220
  Train screens: 53776  |  Val screens: 5975

[Step 4/5] Model: 13,308,314 parameters (spatial=on)

[Step 5/5] Training: 120 epochs, batch_size=256, lr=0.001, full-vocabulary softmax


C:\Users\Jonathan\Dev\screen2vec-revisited\component_utils.py:312: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, n_layers)


Epoch   1/120 | Train: 12.3212 (text 11.6266 / class 0.6946) | Val: 12.0637 * best


Epoch   2/120 | Train: 11.9168 (text 11.3705 / class 0.5463) | Val: 11.9199 * best


... (5 outputs left out)

Epoch   8/120 | Train: 11.3551 (text 10.8650 / class 0.4901) | Val: 11.4367 (patience 1/15)


Epoch   9/120 | Train: 11.3159 (text 10.8311 / class 0.4848) | Val: 11.3533 * best


Epoch  10/120 | Train: 11.2594 (text 10.7798 / class 0.4796) | Val: 11.3521 * best


Epoch 11/120:  74%|███████▍  | 156/211 [24:16<21:46, 23.75s/it]

In [ ]:
%run 13_evaluate_component_prediction.py --data_dir component_data --model component_predictor_euclidean.pth

In [ ]:
%run 13_evaluate_component_prediction.py --data_dir component_data --model component_predictor_euclidean.pth --eval_split all --label ">>> Ours-EUCLIDEAN (all screens) <<<"

In [ ]:
# 4. HIERARCHICAL context — the second row of Table 2

In [ ]:
%run 12_train_component_predictor.py --data_dir component_data --distance hierarchical --plot

In [ ]:
%run 13_evaluate_component_prediction.py --data_dir component_data --model component_predictor_hierarchical.pth --label ">>> Ours-HIERARCHICAL <<<"

In [ ]:
%run 13_evaluate_component_prediction.py --data_dir component_data --model component_predictor_hierarchical.pth --eval_split all --label ">>> Ours-HIERARCHICAL (all screens) <<<"

In [ ]:
# 5. Reference points: untrained baseline, and the paper's information set

In [ ]:
%run 13_evaluate_component_prediction.py --data_dir component_data --model none --eval_split all --label ">>> Mean-of-context baseline <<<"

In [ ]:
%run 12_train_component_predictor.py --data_dir component_data --distance euclidean --no_spatial --output component_predictor_euclidean_nospatial.pth

In [ ]:
%run 13_evaluate_component_prediction.py --data_dir component_data --model component_predictor_euclidean_nospatial.pth --label ">>> Ours-EUCLIDEAN (no spatial) <<<"

## Protocol notes

The task (paper Sec. 2.5): for every GUI component carrying text, predict that
text from the 16 components closest to it on the same screen, ranked against the
embeddings of **all** component texts in RICO. The two Table 2 rows differ only
in how "closest" is measured — bounding-box gap (EUCLIDEAN) vs. view-hierarchy
edge count (HIERARCHICAL).

| script | role |
|---|---|
| `10_extract_components.py` | cleans each screen with `utils.clean_screen()`, flattens it into components (text, one of the 26 Screen2Vec class types, binned bounds, parent), collects the vocabulary |
| `11_embed_component_vocab.py` | embeds every unique component text — replaces Screen2Vec's Sentence-BERT `vocab_emb.npy` |
| `12_train_component_predictor.py` | Transformer context encoder in place of `HiddenLabelPredictorModel`; text CE over the whole vocabulary + class CE, Adam lr 1e-3, batch 256 |
| `13_evaluate_component_prediction.py` | Table 2 metrics and the comparison table |

`component_utils.py` holds what they share: the class taxonomy (paper Table 1),
component extraction, both distance measures, and the model.

Where this differs from the original, and why:

1. **No self-leak in the context.** `dataset.py::get_closest_UI_obj` falls back
   to `[*range(len(labeled_uis))]` when a screen has ≤ 16 components, putting the
   target *into its own context*. We always exclude it, so for those screens the
   published 0.443 / 0.588 is an easier task than ours.
2. **Held-out by default.** `modeltester.py` scores whatever dataset path it is
   given, training screens included. Default here is the 10% validation split;
   the `--eval_split all` cells give the paper's setting.
3. **Components come from `utils.clean_screen()`** — the same cleaned
   representation the screen embeddings are built from (~23 components per
   screen, ~9 with text), against Screen2Vec's every-visible-node list.
4. **Ranking happens in a PCA-reduced space** (`--pred_dim`, 512 by default),
   which is what makes a full-vocabulary softmax affordable at 4096 dimensions.
   `--pred_dim 0` keeps the encoder's full dimensionality; `--neg_vocab 8192`
   switches to a sampled softmax if the full one does not fit.
5. **Spatial features are on by default** (context box, its size, its offset from
   the target box); the `--no_spatial` cell restricts the model to Screen2Vec's
   own information set, where positions only *order* the context.
6. **Screen embeddings are deliberately not used as context** — they are computed
   from the whole screen JSON, target text included, so feeding one in would leak
   the answer.

Knobs worth trying if the numbers fall short: `--targets_per_screen 4` (each
screen has ~9 text-bearing components but the paper's protocol hides one per
epoch), `--pred_dim 1024`, `--d_model 768 --n_layers 6`.